## 1. Imports

In [1]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

from sklearn.model_selection import (
    TimeSeriesSplit, 
    cross_val_score
)
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor

## 2. Load data

In [2]:
DF_PATH = Path("../data/processed/hour_feature_engineered.parquet")
df = pd.read_parquet(DF_PATH)

# look at the DataFrame
df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,...,year,day,hour_sin,hour_cos,month_sin,month_cos,cnt_lag_1,cnt_lag_24,cnt_lag_168,cnt
0,2011-01-08,1,0,1,7,0,6,0,2,0.16,...,2011,8,0.965926,-0.258819,0.5,0.866025,2.0,84.0,16.0,9
1,2011-01-08,1,0,1,8,0,6,0,3,0.16,...,2011,8,0.866025,-0.500000,0.5,0.866025,9.0,210.0,40.0,15
2,2011-01-08,1,0,1,9,0,6,0,3,0.16,...,2011,8,0.707107,-0.707107,0.5,0.866025,15.0,134.0,32.0,20
3,2011-01-08,1,0,1,10,0,6,0,2,0.18,...,2011,8,0.500000,-0.866025,0.5,0.866025,20.0,63.0,13.0,61
4,2011-01-08,1,0,1,11,0,6,0,2,0.20,...,2011,8,0.258819,-0.965926,0.5,0.866025,61.0,67.0,1.0,62


## 3. Train test split

In [3]:
# check min and max dates
print(f"Min: {df["dteday"].min()}")
print(f"Max: {df["dteday"].max()}")

Min: 2011-01-08 00:00:00
Max: 2012-12-31 00:00:00


In [4]:
train = df[(df["dteday"] >= pd.Timestamp(2011, 1, 8)) & (df["dteday"] <= pd.Timestamp(2012, 9, 30))]
valid = df[df["dteday"] > pd.Timestamp(2012, 9, 30)]

# drop dteday as well as we don't need it for modeling
X_train = train.drop(columns=["dteday", "cnt"])
y_train = train["cnt"]
X_test = valid.drop(columns=["dteday", "cnt"])
y_test = valid["cnt"]

## 4. Identify columns

In [5]:
# I'm not including year and yr because they are redundant
numerical_columns = ["temp", "atemp", "hum", "windspeed", "day", "hour_sin", 
                     "hour_cos", "month_sin", "month_cos", "cnt_lag_1", "cnt_lag_24", "cnt_lag_168"]
categorical_columns = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]

print(f"Numerical columns count: {len(numerical_columns)}")
print(f"Categorical columns count: {len(categorical_columns)}")

Numerical columns count: 12
Categorical columns count: 7


## 5. Build preprocessing pipelines

In [6]:
num_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

## 6. ColumnTransformer

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", num_pipeline, numerical_columns),
        ("categorical", cat_pipeline, categorical_columns),
    ]
)

## 7. Create models

In [8]:
models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(n_jobs=-1),
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1),
    "XG Boost": XGBRegressor(
        random_state=42,
        n_jobs=-1),
}

## 8. Train every model

In [9]:
results = []
tscv = TimeSeriesSplit(n_splits=5)

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=tscv,
        scoring="r2"
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "CV Mean R²": cv_scores.mean(),
        "CV Std R²": cv_scores.std(),
        "MAE": mae,
        "RMSE": rmse,
        "R² score": r2,
    })

# look at the results
results_df = pd.DataFrame(results).sort_values(by="R² score").round(2)
results_df

,Model,CV Mean R²,CV Std R²,MAE,RMSE,R² score
0,Dummy,-0.22,0.17,156.94,204.19,-0.03
1,Linear Regression,0.86,0.02,50.48,74.45,0.86
2,Random Forest,0.89,0.04,30.53,53.65,0.93
3,XG Boost,0.90,0.03,31.50,50.81,0.94


- The Dummy Regressor performed substantially worse than all other models, confirming that the engineered features contain significant predictive information.

- Linear Regression achieved an R² score of 0.86, indicating that a large portion of bike demand variability can be explained through approximately linear relationships between the predictors and the target.

- Tree-based ensemble methods substantially outperformed Linear Regression, suggesting the presence of nonlinear interactions and threshold effects in the data.

- XGBoost achieved the best overall performance with an R² score of 0.94 and the lowest RMSE (50.81), while Random Forest obtained the lowest MAE (30.53).

- The lower RMSE obtained by XGBoost suggests that it handled large prediction errors more effectively, whereas Random Forest achieved slightly better average prediction accuracy.

## 9. Save the best model

In [10]:
best_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            random_state=42,
            n_jobs=-1)),
    ]
)
best_model.fit(X_train, y_train)

MODEL_PATH = Path("../models/xg_boost_baseline.pkl")
joblib.dump(best_model, MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")

Model saved to ..\models\xg_boost_baseline.pkl
